In [ ]:
function clickConnect() {
  document.querySelector('#top-toolbar > colab-connect-button')
    ?.shadowRoot?.querySelector('div')?.click();
}
setInterval(clickConnect, 60000);

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU — Runtime → Change runtime type → T4 GPU")

In [ ]:
!pip install transformers datasets sentencepiece sacrebleu torch evaluate scikit-learn sacremoses -q
print("✅ Done!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

In [ ]:
from google.colab import files
print("Upload your burushaski_8000.csv")
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv(
    'burushaski_8000.csv',
    on_bad_lines='skip',
    encoding='utf-8'
)
print(f"Rows: {len(df)}")
print(f"Columns before: {df.columns.tolist()}")

df.columns = [col.strip().lower() for col in df.columns]
df = df.rename(columns={
    'english': 'text',
    'burushaski': 'target'
})
df = df[['text', 'target']]
df = df.dropna().drop_duplicates()

df.to_csv('burushaski_fixed.csv', index=False)
print(f"Columns after: {df.columns.tolist()}")
print(f"Total rows: {len(df)}")
print(df.head(3))
print("✅ CSV fixed!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import torch
import evaluate
print("✅ All libraries loaded!")

In [ ]:
CSV_FILE       = "burushaski_fixed.csv"
ENGLISH_COL    = "text"
BURUSHASKI_COL = "target"
OUTPUT_DIR     = "/content/drive/MyDrive/burushaski_model"
MODEL_NAME     = "Helsinki-NLP/opus-mt-en-mul"
MAX_LENGTH     = 128
BATCH_SIZE     = 16
EPOCHS         = 30
LEARNING_RATE  = 3e-5
print("✅ Settings ready!")
print(f"✅ Saving to: {OUTPUT_DIR}")

In [ ]:
df = pd.read_csv(
    CSV_FILE,
    on_bad_lines='skip',
    encoding='utf-8'
)
df = df[[ENGLISH_COL, BURUSHASKI_COL]].dropna().drop_duplicates()
df[ENGLISH_COL]    = df[ENGLISH_COL].astype(str).str.strip()
df[BURUSHASKI_COL] = df[BURUSHASKI_COL].astype(str).str.strip()
df = df[df[ENGLISH_COL].str.split().str.len() >= 2]
df = df[df[BURUSHASKI_COL].str.split().str.len() >= 2]
df = df[df[ENGLISH_COL].str.split().str.len() <= 100]
df = df[df[BURUSHASKI_COL].str.split().str.len() <= 100]
print(f"✅ Clean rows: {len(df)}")

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=42
)
print(f"✅ Train: {len(train_df)} | Test: {len(test_df)}")

In [ ]:
train_data = Dataset.from_dict({
    "en":  train_df[ENGLISH_COL].tolist(),
    "bur": train_df[BURUSHASKI_COL].tolist()
})
test_data = Dataset.from_dict({
    "en":  test_df[ENGLISH_COL].tolist(),
    "bur": test_df[BURUSHASKI_COL].tolist()
})
dataset = DatasetDict({
    "train": train_data,
    "test":  test_data
})
print("✅ Dataset created!")

In [ ]:
print("Loading model...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model     = model.to(device)
print(f"✅ Model loaded! Using: {device}")

In [ ]:
def preprocess(examples):
    model_inputs = tokenizer(
        examples["en"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        examples["bur"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["en", "bur"]
)
print("✅ Tokenizing done!")

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True
)
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(
        preds, skip_special_tokens=True
    )
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]
    result = metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return {"bleu": round(result["score"], 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    num_train_epochs            = EPOCHS,
    weight_decay                = 0.01,
    save_total_limit            = 3,
    predict_with_generate       = True,
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 50,
    load_best_model_at_end      = True,
    metric_for_best_model       = "bleu",
    greater_is_better           = True,
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized["train"],
    eval_dataset    = tokenized["test"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(
        early_stopping_patience=3
    )]
)
print("✅ Trainer ready!")

In [ ]:
print("🚀 Training started!")
print("✅ Saving to Google Drive after every epoch")
print("✅ Safe from disconnection")
print("-" * 50)
trainer.train()
print("-" * 50)
print("✅ Training complete!")